In [1]:
from ultralytics import YOLO
from roboflow import Roboflow
from dotenv import dotenv_values, load_dotenv
import shutil
import os
import torch

## Get Dataset

In [2]:
secrets = dotenv_values("../.env")

rf = Roboflow(api_key=secrets["ROBOFLOW_API_KEY"])
project = rf.workspace("workspace-5ujvu").project("basketball-players-fy4c2-vfsuv")
version = project.version(17)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Basketball-Players-17 in yolo26:: 100%|██████████| 652/652 [00:00<00:00, 17577.93it/s]


In [3]:
dataset.location

'/home/roy/dev/projects/ai/basketball-analysis/training/Basketball-Players-17'

## Move Dataset

Have to move test, train etc into Basketball-Players-17/Basketball-Players-17. It's the format YOLO requires for YOLOV5 newer ones usually don't need it but no harm in doing it apart from disk space.

In [4]:

if not os.path.exists(f"{dataset.location}/Basketball-Players-17/train"):
    shutil.copytree(f"{dataset.location}/train", f"{dataset.location}/Basketball-Players-17/train")

if not os.path.exists(f"{dataset.location}/Basketball-Players-17/test"):
    shutil.copytree(f"{dataset.location}/test", f"{dataset.location}/Basketball-Players-17/test")

if not os.path.exists(f"{dataset.location}/Basketball-Players-17/valid"):
    shutil.copytree(f"{dataset.location}/valid", f"{dataset.location}/Basketball-Players-17/valid")

## Training

In [ ]:
import gc 

# Optional Cleanup to free up memory before training
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

device = "cuda:0" if torch.cuda.is_available() else "cpu"

model = YOLO("models/yolo26x.pt")

model.train(
    task="detect",
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    workers=4,
    device=device
)

Ultralytics 8.4.38 🚀 Python-3.14.2 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 15842MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/roy/dev/projects/ai/basketball-analysis/training/Basketball-Players-17/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=models/yolo26x.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False